In [5]:
!pip install -q pandas pythainlp jiwer


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import pandas as pd
import re

df_truth = pd.read_csv("Categorized-Data/Gowajee-Corpus/thai_foreign/final_data.csv")
df_small = pd.read_csv("Gowajee-Thai-Foreign-Whisper-Small.csv", header=0, names=['pred'])
df_large = pd.read_csv("Gowajee-Thai-Foreign-Whisper-Large.csv", header=0, names=['pred'])
df_seamless = pd.read_csv("seamless_m4t_gowajee_results.csv")

print(f"Loaded: Truth={len(df_truth)}, Small={len(df_small)}, Large={len(df_large)}, Seamless={len(df_seamless)}")

Loaded: Truth=957, Small=957, Large=957, Seamless=957


In [2]:
def clean_whisper(text):
    text = str(text)
    if 'text:' in text:
        after_text = text.split('text:')[1].strip()
        match = re.search(r'^(.*?)(?:\n|\s+(?:\w+|\S+),\s*prob:|$)', after_text)
        if match:
            return match.group(1).strip()
    return text.strip()

df_truth['small_pred'] = df_small['pred'].apply(clean_whisper)
df_truth['large_pred'] = df_large['pred'].apply(clean_whisper)
df_truth['seamless_pred'] = df_seamless['hypothesis'].fillna("")

In [6]:
import jiwer
from pythainlp.tokenize import word_tokenize

# --- ฟังก์ชันคำนวณ EWA ---
def calculate_ewa(truth_list, pred_list):
    total_eng_words = 0
    correct_eng_words = 0
    for t, p in zip(truth_list, pred_list):
        t_words = re.findall(r'[a-zA-Z]+', str(t).lower())
        p_words = re.findall(r'[a-zA-Z]+', str(p).lower())
        total_eng_words += len(t_words)
        for w in t_words:
            if w in p_words:
                correct_eng_words += 1
                p_words.remove(w)
    return (correct_eng_words / total_eng_words) * 100 if total_eng_words > 0 else 0

# --- ฟังก์ชันคำนวณ WER (Word Error Rate) ---
def calculate_wer(truth_list, pred_list):
    wer_scores = []
    for t, p in zip(truth_list, pred_list):
        # ใช้ PyThaiNLP ตัดคำไทยให้แยกออกจากกันด้วย Spacebar ก่อน
        t_tokens = " ".join(word_tokenize(str(t), engine='newmm'))
        p_tokens = " ".join(word_tokenize(str(p), engine='newmm'))
        
        # ข้ามประโยคที่ Ground Truth ว่างเปล่า
        if not t_tokens.strip():
            continue
            
        try:
            score = jiwer.wer(t_tokens, p_tokens)
            wer_scores.append(score)
        except ValueError:
            pass
            
    # แปลงเป็นเปอร์เซ็นต์ (ยิ่งน้อยยิ่งดี)
    return (sum(wer_scores) / len(wer_scores)) * 100 if wer_scores else 0

# --- คำนวณและแสดงผล ---
ewa_small = calculate_ewa(df_truth['text_cleaned'], df_truth['small_pred'])
ewa_large = calculate_ewa(df_truth['text_cleaned'], df_truth['large_pred'])
ewa_seamless = calculate_ewa(df_truth['text_cleaned'], df_truth['seamless_pred'])

wer_small = calculate_wer(df_truth['text_cleaned'], df_truth['small_pred'])
wer_large = calculate_wer(df_truth['text_cleaned'], df_truth['large_pred'])
wer_seamless = calculate_wer(df_truth['text_cleaned'], df_truth['seamless_pred'])

print("📊 สรุปผลการประเมินประสิทธิภาพ (Metrics)")
print("=" * 60)
print(f"1. Whisper Large -> EWA: {ewa_large:.2f}% | WER: {wer_large:.2f}%")
print(f"2. Whisper Small -> EWA: {ewa_small:.2f}% | WER: {wer_small:.2f}%")
print(f"3. SeamlessM4T   -> EWA: {ewa_seamless:.2f}% | WER: {wer_seamless:.2f}%")
print("=" * 60)
print("* หมายเหตุ: EWA ยิ่งมากยิ่งดี / WER ยิ่งน้อยยิ่งดี")

📊 สรุปผลการประเมินประสิทธิภาพ (Metrics)
1. Whisper Large -> EWA: 70.86% | WER: 47.70%
2. Whisper Small -> EWA: 46.34% | WER: 66.24%
3. SeamlessM4T   -> EWA: 11.11% | WER: 64.69%
* หมายเหตุ: EWA ยิ่งมากยิ่งดี / WER ยิ่งน้อยยิ่งดี


In [4]:
count = 0
for i in range(len(df_truth)):
    truth_text = str(df_truth['text_cleaned'].iloc[i])
    s_text = str(df_truth['small_pred'].iloc[i])
    l_text = str(df_truth['large_pred'].iloc[i])
    se_text = str(df_truth['seamless_pred'].iloc[i])
    
    t_words = re.findall(r'[a-zA-Z]+', truth_text.lower())
    s_words = re.findall(r'[a-zA-Z]+', s_text.lower())
    l_words = re.findall(r'[a-zA-Z]+', l_text.lower())
    se_words = re.findall(r'[a-zA-Z]+', se_text.lower())
    
    if not t_words:
        continue
        
    s_missing = [w for w in t_words if w not in s_words]
    l_missing = [w for w in t_words if w not in l_words]
    se_missing = [w for w in t_words if w not in se_words]
    
    if not l_missing and (s_missing or se_missing):
        print(f"Truth: {truth_text}")
        print(f"Target English Words: {t_words}")
        print(f"Whisper Small: {s_text} | Missing: {s_missing}")
        print(f"Whisper Large: {l_text} | Missing: {l_missing}")
        print(f"SeamlessM4T: {se_text} | Missing: {se_missing}")
        print("-" * 50)
        count += 1
        
    if count >= 5:
        break

Truth: โกวาจี เล่น เพลง Shape of You
Target English Words: ['shape', 'of', 'you']
Whisper Small: กองวาจีเล่นเพลง cheapabyou | Missing: ['shape', 'of', 'you']
Whisper Large: โกวาจี เล่นเพลง shape of you | Missing: []
SeamlessM4T: โกวาจิเล่นเพลง Shape of You | Missing: []
--------------------------------------------------
Truth: โกวาจี เล่น เพลง Sugar
Target English Words: ['sugar']
Whisper Small: ก็ว่าจี่ เล่นเพลง chill คา | Missing: ['sugar']
Whisper Large: โกวาจี เล่นเพลง sugar | Missing: []
SeamlessM4T: โกวาจี เล่นเพลงชูกะ | Missing: ['sugar']
--------------------------------------------------
Truth: โกวาจี เล่น เพลง Stay
Target English Words: ['stay']
Whisper Small: กว่าจีเล่นเพลง สติ | Missing: ['stay']
Whisper Large: ก็ว่าที่เล่นเพลง stay | Missing: []
SeamlessM4T: โควาจิเล่นเพลงสเตย์ | Missing: ['stay']
--------------------------------------------------
Truth: เพิ่ม ใน playlist เพลง โปรด
Target English Words: ['playlist']
Whisper Small: เพิ่งป่อน | Missing: ['playlist']
Whisper L